# Projet Final : Chatbot Documentation Technique

## Contexte du Projet

Vous etes charge de creer un chatbot intelligent pour repondre aux questions sur une documentation technique (Python, Machine Learning, ou autre domaine de votre choix).

## Objectifs du Projet

1. **Ingestion** : Charger et decouper des documents de documentation
2. **Indexation** : Creer un vector store optimise
3. **Retrieval** : Implementer une recherche semantique performante
4. **Generation** : Generer des reponses precises avec citations
5. **Evaluation** : Evaluer avec RAGAS sur un dataset de test
6. **API** : Creer une API FastAPI production-ready


## Installation

In [ ]:
# Installation des dependances
# !pip install langchain langchain-community langchain-openai chromadb sentence-transformers
# !pip install ragas datasets
# !pip install fastapi uvicorn pydantic
# !pip install numpy pandas matplotlib seaborn

In [1]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from typing import List, Dict, Optional, Tuple
from datetime import datetime
import logging

# Configuration
os.environ["MISTRAL_API_KEY"] = "votre-cle-api"  # REMPLACER PAR VOTRE CLE

# Logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

## Etape 1 : Preparation des Documents 



In [2]:
documents_solution = [
    "Python est un langage de programmation interprete, interactif et oriente objet. Il a ete cree par Guido van Rossum et publie pour la premiere fois en 1991. Python est connu pour sa syntaxe claire et lisible, ce qui en fait un excellent choix pour les debutants. Le langage supporte plusieurs paradigmes de programmation, notamment la programmation procedurale, orientee objet et fonctionnelle.",
    "Les listes en Python sont des collections ordonnees et modifiables. On les cree avec des crochets: ma_liste = [1, 2, 3]. Les listes peuvent contenir des elements de differents types. On peut ajouter des elements avec append(), inserer avec insert(), et supprimer avec remove() ou pop(). Les listes supportent l'indexation negative, le slicing, et de nombreuses methodes utiles comme sort(), reverse(), et extend().",
    "Les dictionnaires Python stockent des paires cle-valeur. On les cree ainsi: mon_dict = {'nom': 'Alice', 'age': 30}. Les cles doivent etre uniques et immuables (strings, nombres, tuples). On accede aux valeurs avec dict['cle'] ou dict.get('cle'). Les dictionnaires sont tres efficaces pour les lookups et sont largement utilises en Python. Ils supportent des operations comme keys(), values(), items(), update(), et pop().",
    "Les fonctions en Python se definissent avec le mot-cle def. Syntaxe: def ma_fonction(param1, param2): return resultat. Les fonctions peuvent avoir des parametres par defaut, des parametres avec nom, et des parametres variables (*args, **kwargs). Python supporte aussi les fonctions lambda (anonymes), les decorateurs, et les closures. Les fonctions sont des objets de premiere classe en Python.",
    "Les classes en Python permettent la programmation orientee objet. On les definit avec le mot-cle class. Une classe contient des attributs (variables) et des methodes (fonctions). Le constructeur __init__ initialise les objets. Python supporte l'heritage (simple et multiple), le polymorphisme, et l'encapsulation. Les attributs peuvent etre publics, proteges (_attr) ou prives (__attr).",
    "Les modules Python sont des fichiers .py contenant du code reutilisable. On les importe avec import ou from module import fonction. Python possede une vaste bibliotheque standard (math, os, sys, datetime, etc.). On peut aussi installer des packages externes avec pip. Les packages sont des collections de modules organises dans des repertoires avec un fichier __init__.py.",
    "La gestion des exceptions en Python utilise try/except/finally. Syntaxe: try: code_risque except ExceptionType: gestion_erreur finally: nettoyage. Python a de nombreuses exceptions built-in comme ValueError, TypeError, KeyError. On peut creer ses propres exceptions en heritant de Exception. Les exceptions permettent de gerer les erreurs proprement sans crasher le programme.",
    "Les comprehensions Python sont des syntaxes concises pour creer des listes, dictionnaires, ou ensembles. Liste: [x*2 for x in range(10) if x%2==0]. Dict: {k: v for k, v in items}. Set: {x for x in data}. Les comprehensions sont plus rapides et lisibles que les boucles for equivalentes. Elles supportent les conditions if et les boucles imbriquees.",
    "Le RAG (Retrieval-Augmented Generation) est une technique qui combine la recherche de documents avec la generation de texte par un LLM. Le systeme recupere d'abord des documents pertinents dans une base de connaissances vectorielle, puis utilise ces documents comme contexte pour generer une reponse precise et ancree dans des sources fiables. Le RAG reduit les hallucinations et permet de citer les sources.",
    "Les embeddings sont des representations vectorielles de texte qui capturent la semantique. Ils sont generes par des modeles d'apprentissage profond comme sentence-transformers ou OpenAI embeddings. Les embeddings transforment du texte en vecteurs denses dans un espace multidimensionnel ou des textes semantiquement similaires sont proches. Ils permettent de calculer la similarite cosinus entre textes.",
    "ChromaDB est une base de donnees vectorielle open-source concue pour les applications d'IA. Elle permet de stocker des embeddings et d'effectuer des recherches de similarite rapidement. ChromaDB supporte la persistance, les metadonnees, les filtres, et peut fonctionner en mode client-serveur. Elle s'integre facilement avec LangChain et d'autres frameworks RAG. ChromaDB utilise HNSW pour les recherches rapides.",
    "LangChain est un framework pour developper des applications basees sur des LLMs. Il fournit des abstractions pour les chains (sequences d'operations), les agents (decision making), les retrievers, et les memory systems. LangChain supporte de nombreux LLMs (OpenAI, Anthropic, HuggingFace) et vector stores. Il facilite la creation de pipelines RAG complexes avec des composants reutilisables.",
    "Le chunking est le processus de decoupage de documents en morceaux plus petits. C'est crucial pour le RAG car les LLMs ont des limites de contexte. Un bon chunking preserve la coherence semantique. Les strategies incluent: fixed-size (taille fixe), recursive (hierarchique), semantic (base sur le sens). Les parametres importants sont chunk_size (taille) et chunk_overlap (chevauchement).",
    "Les metriques d'evaluation RAG incluent: Precision@K (proportion de docs pertinents dans top-K), Recall@K (proportion de docs pertinents trouves), MRR (Mean Reciprocal Rank), faithfulness (fidelite au contexte), answer relevancy (pertinence de la reponse). RAGAS est un framework populaire pour evaluer les systemes RAG avec des metriques automatiques basees sur des LLMs.",
    "FastAPI est un framework web moderne pour Python. Il permet de creer des APIs REST performantes avec validation automatique via Pydantic, documentation OpenAPI automatique, et support async/await. FastAPI est ideal pour deployer des modeles ML et des systemes RAG en production. Il supporte les middlewares, l'authentification, et s'integre bien avec uvicorn pour le serving.",
]

metadatas_solution = [
    {
        "source": "python_intro.md",
        "topic": "python",
        "subtopic": "basics",
        "difficulty": "easy",
    },
    {
        "source": "python_lists.md",
        "topic": "python",
        "subtopic": "data_structures",
        "difficulty": "easy",
    },
    {
        "source": "python_dicts.md",
        "topic": "python",
        "subtopic": "data_structures",
        "difficulty": "easy",
    },
    {
        "source": "python_functions.md",
        "topic": "python",
        "subtopic": "functions",
        "difficulty": "medium",
    },
    {
        "source": "python_classes.md",
        "topic": "python",
        "subtopic": "oop",
        "difficulty": "medium",
    },
    {
        "source": "python_modules.md",
        "topic": "python",
        "subtopic": "modules",
        "difficulty": "medium",
    },
    {
        "source": "python_exceptions.md",
        "topic": "python",
        "subtopic": "exceptions",
        "difficulty": "medium",
    },
    {
        "source": "python_comprehensions.md",
        "topic": "python",
        "subtopic": "advanced",
        "difficulty": "medium",
    },
    {
        "source": "rag_intro.md",
        "topic": "ml",
        "subtopic": "rag",
        "difficulty": "medium",
    },
    {
        "source": "embeddings.md",
        "topic": "ml",
        "subtopic": "embeddings",
        "difficulty": "medium",
    },
    {
        "source": "chromadb.md",
        "topic": "ml",
        "subtopic": "vectorstores",
        "difficulty": "medium",
    },
    {
        "source": "langchain.md",
        "topic": "ml",
        "subtopic": "frameworks",
        "difficulty": "medium",
    },
    {"source": "chunking.md", "topic": "ml", "subtopic": "rag", "difficulty": "hard"},
    {
        "source": "evaluation.md",
        "topic": "ml",
        "subtopic": "metrics",
        "difficulty": "hard",
    },
    {
        "source": "fastapi.md",
        "topic": "ml",
        "subtopic": "deployment",
        "difficulty": "medium",
    },
]

print(f"{len(documents_solution)} documents prepares")
print(f"{len(metadatas_solution)} metadonnees preparees")

15 documents prepares
15 metadonnees preparees


## Etape 2 : Ingestion et Chunking

### Objectif
Implementer un systeme d'ingestion robuste avec chunking optimise.

### Instructions

1. Creer une classe `DocumentProcessor` qui :
   - Charge les documents
   - Les decoupe en chunks optimaux
   - Ajoute des metadonnees enrichies

2. **Parametres a tester** :
   - Taille de chunk : 300, 500, 800 caracteres
   - Overlap : 30, 50, 100 caracteres
   - Separateurs personnalises

3. **Analyser les chunks** :
   - Nombre total de chunks
   - Taille moyenne des chunks
   - Distribution des tailles

### A Faire
Completez la classe ci-dessous :

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document


class DocumentProcessor:
    """
    TODO: Implementer la classe de traitement des documents
    """

    def __init__(self, chunk_size: int = 500, chunk_overlap: int = 50):
        # TODO: Initialiser le text splitter
        pass

    def process(self, texts: List[str], metadatas: List[Dict]) -> List[Document]:
        """
        TODO: Traiter les documents et retourner les chunks
        """
        pass

    def analyze_chunks(self, chunks: List[Document]) -> Dict:
        """
        TODO: Analyser les statistiques des chunks
        """
        pass


# TODO: Tester votre processor
# processor = DocumentProcessor(chunk_size=500, chunk_overlap=50)
# chunks = processor.process(documents, metadatas)
# stats = processor.analyze_chunks(chunks)

## Etape 3 : Vector Store et Retrieval

### Objectif
Creer un systeme de retrieval performant avec ChromaDB.

### Instructions

1. Creer une classe `VectorStoreManager` qui :
   - Initialise ChromaDB avec persistance
   - Indexe les documents
   - Implemente plusieurs strategies de recherche

2. **Strategies de retrieval a implementer** :
   - Similarity search classique
   - MMR (Maximum Marginal Relevance)

3. **Tester et comparer** :
   - Tester avec 5 questions
   - Comparer les resultats des differentes strategies
   - Analyser les scores de similarite

### A Faire

In [ ]:
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings


class VectorStoreManager:
    """
    TODO: Implementer la classe de gestion du vector store
    """

    def __init__(self, persist_dir: str = "./project_chroma"):
        # TODO: Initialiser
        pass

    def create_index(self, documents: List[Document]):
        # TODO: Creer l'index
        pass

    def similarity_search(self, query: str, k: int = 4) -> List[Document]:
        # TODO: Recherche classique
        pass

    def mmr_search(self, query: str, k: int = 4) -> List[Document]:
        # TODO: Recherche MMR
        pass


# TODO: Tester votre vector store

## Etape 4 : Generation et RAG Chain

### Objectif
Implementer le pipeline de generation avec prompt engineering.

### Instructions

1. Creer une classe `RAGPipeline` qui :
   - Orchestration retrieval + generation
   - Prompt engineering avec citations
   - Gestion des erreurs

2. **Prompt engineering** :
   - Le LLM doit citer ses sources
   - Il doit dire "je ne sais pas" si info non disponible
   - Reponses claires et structurees

3. **Tester** :
   - Questions faciles (info directement dans les docs)
   - Questions complexes (info dispersee)
   - Questions hors sujet

### A Faire

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate


class RAGPipeline:
    """
    TODO: Implementer le pipeline RAG complet
    """

    def __init__(self, vectorstore_manager: VectorStoreManager):
        # TODO: Initialiser LLM et prompt
        pass

    def answer(
        self,
        question: str,
        k: int = 4,
        use_mmr: bool = False,
        filters: Optional[Dict] = None,
    ) -> Dict:
        """
        TODO: Pipeline complet retrieval + generation

        Returns:
            Dict avec answer, sources, metadata
        """
        pass


# TODO: Tester votre pipeline

## Etape 5 : Dataset de Test

### Objectif
Creer un dataset de test representatif pour l'evaluation.

### Instructions

1. **Creer 20+ questions** couvrant :
   - Questions factuelles (qui/quoi/quand)
   - Questions how-to (comment faire X)
   - Questions conceptuelles (expliquer Y)
   - Questions hors sujet (pour tester robustesse)

2. **Pour chaque question** :
   - Reponse attendue (ground truth)
   - Documents pertinents
   - Categorie et difficulte

3. **Format JSON** :
```json
{
  "id": 1,
  "question": "...",
  "expected_answer": "...",
  "relevant_docs": [...],
  "category": "factual",
  "difficulty": "easy"
}
```

### A Faire

In [ ]:
# TODO: Creer votre dataset de test

test_dataset = [
    # Minimum 20 questions
    {
        "id": 1,
        "question": "...",
        "expected_answer": "...",
        "relevant_docs": [...],
        "category": "factual",
        "difficulty": "easy",
    },
    # TODO: Ajouter 19+ autres questions
]

# Verification
assert len(test_dataset) >= 20, "Minimum 20 questions requises"

# Statistiques
df_test = pd.DataFrame(test_dataset)
print(f"{len(test_dataset)} questions creees")
print(f"\nCategories: {df_test['category'].value_counts().to_dict()}")
print(f"Difficultes: {df_test['difficulty'].value_counts().to_dict()}")

# Sauvegarder
with open("test_dataset.json", "w", encoding="utf-8") as f:
    json.dump(test_dataset, f, ensure_ascii=False, indent=2)

## Etape 6 : Evaluation avec RAGAS

### Objectif
Evaluer le systeme avec des metriques quantitatives.

### Instructions

1. **Executer le pipeline** sur tout le dataset de test
2. **Evaluer avec RAGAS** :
   - Faithfulness
   - Answer Relevancy
   - Context Precision
   - Context Recall

3. **Analyser les resultats** :
   - Scores moyens par metrique
   - Scores par categorie

### A Faire

In [ ]:
from ragas import evaluate
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_precision,
    context_recall,
)
from datasets import Dataset

# TODO: Executer le pipeline sur le dataset de test


def evaluate_rag_system(rag_pipeline, test_dataset):
    """
    TODO: Evaluer le systeme RAG complet
    """
    results = []

    # Pour chaque question
    for item in test_dataset:
        # TODO: Obtenir la reponse du RAG
        # TODO: Formater pour RAGAS
        pass

    # TODO: Evaluer avec RAGAS

    return results


# TODO: Lancer l'evaluation
# evaluation_results = evaluate_rag_system(rag_pipeline, test_dataset)

## Etape 7 : API FastAPI

### Objectif
Creer une API production-ready.

### Instructions

1. **Endpoints a creer** :
   - `GET /` : Info sur l'API
   - `GET /health` : Health check
   - `POST /ask` : Poser une question

2. **Fonctionnalites** :
   - Validation des inputs avec Pydantic
   - Gestion des erreurs
   - Logging des requetes

3. **Tester** :
   - Requetes valides
   - Requetes invalides

### A Faire

In [ ]:
# TODO: Creer votre API FastAPI
# Le code doit etre mis dans un fichier Python separe : api.py

api_template = """
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from typing import List, Dict, Optional

# TODO: Importer vos classes (RAGPipeline, VectorStoreManager, etc.)

app = FastAPI(title="Documentation Chatbot API")

# TODO: Implementer les endpoints

@app.get("/")
async def root():
    # TODO
    pass

@app.get("/health")
async def health():
    # TODO
    pass

@app.post("/ask")
async def ask(question: str):
    # TODO
    pass

if __name__ == "__main__":
    import uvicorn
    uvicorn.run(app, host="0.0.0.0", port=8000)
"""

print("Template d'API fourni. A implementer dans un fichier separe.")